In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 30. ZeROテキスト FSDP — テキスト テキスト テキスト

> **学習目標**
> - ZeRO Stage 1/2/3テキスト テキスト テキスト テキスト テキスト
> - FSDP (Fully Sharded Data Parallel)テキスト テキスト テキスト テキスト
> - テキスト テキスト テキスト テキスト 計算テキスト

## 30.1 DDPテキスト テキスト テキスト

DDP: テキスト GPUテキスト モデル テキスト テキスト. 70B モデルテキスト テキスト GPUテキスト テキスト テキスト.

ZeROテキスト テキスト: **"テキスト テキスト テキスト"** — テキスト テキスト, テキスト, テキスト.

## 30.2 テキスト テキスト

学習 テキスト = テキスト + テキスト + テキスト テキスト + テキスト値

AdamW テキスト:
- テキスト $P$ (FP16): $2P$ bytes
- テキスト (FP16): $2P$ bytes
- テキスト テキスト (FP32: m, v, master): $12P$ bytes
- テキスト (テキスト値 テキスト): $16P$ bytes

7B モデル: 112 GB (テキスト GPU テキスト)

## 30.3 ZeRO Stage 1 — テキスト テキスト テキスト

テキスト テキスト $N$テキスト GPUテキスト テキスト:
- テキスト: $16P / N + 4P$ (テキスト + テキスト テキスト)
- 7B, N=8: $16P/8 + 4P = 2P + 4P = 6P = 42GB$ (テキスト テキスト)

## 30.4 ZeRO Stage 2 — テキスト度 テキスト

テキスト度 テキスト:
- テキスト: $16P/N + 2P = 4P$ (8B + 4P + 4P)
- Reduce-Scatter テキスト (All-Reduce テキスト)

## 30.5 ZeRO Stage 3 — テキスト テキスト

テキスト テキスト:
- テキスト: $16P/N$ + テキスト値
- 7B, N=8: $2P + テキスト ≈ 14GB$ → テキスト 40GB GPU テキスト
- テキスト: All-Gather (順伝播), Reduce-Scatter (バックプロパゲーション)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_memory_gb(model_params_b, n_gpus, stage, optimizer_factor=12, dtype_bytes=2):
    """ZeRO stageテキスト Memory (GB).
    model_params_b: Parameter Count (テキスト: 10テキスト)
    optimizer_factor: Adamテキスト テキスト 12 (m, v, master FP32)
    dtype_bytes: 2 for FP16
    """
    P = model_params_b  # 10テキスト テキスト
    # FP16 テキスト: 2 bytes, FP16 grad: 2 bytes
    # Optimizer (FP32 m, v, master): 12 bytes per param
    param_mem = P * dtype_bytes  # GB
    grad_mem = P * dtype_bytes
    opt_mem = P * optimizer_factor

    if stage == 0:  # DDP
        return param_mem + grad_mem + opt_mem
    elif stage == 1:  # Optimizerテキスト テキスト
        return param_mem + grad_mem + opt_mem / n_gpus
    elif stage == 2:  # Optimizer + Gradient テキスト
        return param_mem + (grad_mem + opt_mem) / n_gpus
    elif stage == 3:  # テキスト テキスト
        return (param_mem + grad_mem + opt_mem) / n_gpus

# Visualization: 7B Model, GPU テキスト
model_size = 7  # 7B
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# テキスト: GPU テキスト
ax = axes[0]
for stage, label, color in [(0, 'DDP', 'blue'), (1, 'ZeRO-1', 'green'),
                             (2, 'ZeRO-2', 'orange'), (3, 'ZeRO-3', 'red')]:
    mems = [zero_memory_gb(model_size, n, stage) for n in n_gpus_list]
    ax.plot(n_gpus_list, mems, 'o-', label=label, linewidth=2, color=color)
ax.axhline(40, color='gray', linestyle='--', label='A100 40GB')
ax.axhline(80, color='gray', linestyle=':', label='A100 80GB')
ax.set_xlabel('GPU テキスト')
ax.set_ylabel('GPU テキスト Memory (GB)')
ax.set_title(f'ZeRO Stageテキスト Memory (7B Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# テキスト: Model Sizeテキスト
ax = axes[1]
for stage, label, color in [(0, 'DDP', 'blue'), (3, 'ZeRO-3', 'red')]:
    for n in [1, 8, 32]:
        sizes = [1, 7, 13, 30, 70, 175]
        mems = [zero_memory_gb(s, n, stage) for s in sizes]
        ax.plot(sizes, mems, 'o-', label=f'{label}, N={n}', linewidth=2)
ax.set_xlabel('Model Size (B params)')
ax.set_ylabel('GPU テキスト Memory (GB)')
ax.set_title('Model Sizeテキスト Memory')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch30_zero_memory.png', dpi=100, bbox_inches='tight')
plt.show()

# テキスト Output
print(f"\n{'モデル':>6} | {'GPU':>4} | {'DDP':>8} | {'ZeRO-1':>8} | {'ZeRO-2':>8} | {'ZeRO-3':>8}")
print("-" * 55)
for size in [7, 13, 70]:
    for n in [1, 8]:
        ddp = zero_memory_gb(size, n, 0)
        z1 = zero_memory_gb(size, n, 1)
        z2 = zero_memory_gb(size, n, 2)
        z3 = zero_memory_gb(size, n, 3)
        print(f"{size:>5}B | {n:>4} | {ddp:>7.1f}G | {z1:>7.1f}G | {z2:>7.1f}G | {z3:>7.1f}G")


## 30.6 PyTorch FSDP

FSDP (Fully Sharded Data Parallel) = PyTorch テキスト ZeRO-3.

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyLargeModel()
model = FSDP(model)  # テキスト テキスト テキスト
```

テキスト:
1. 順伝播 テキスト: All-Gatherテキスト テキスト テキスト
2. 順伝播 テキスト: テキスト テキスト テキスト
3. バックプロパゲーション: テキスト All-Gather, テキスト 計算
4. Reduce-Scatterテキスト テキスト テキスト

## 30.7 CPU Offloading

GPU テキスト テキスト テキスト テキスト テキスト CPU RAMテキスト テキスト.
- ZeRO-Infinity, FSDP with CPU offload
- 速度テキスト テキスト テキスト モデル 学習 テキスト


In [ ]:
# FSDP テキスト (gradient sharding)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FSDPSimulator:
    """テキスト GPUテキスト FSDPテキスト テキスト テキスト."""
    def __init__(self, model, n_shards=4):
        self.model = model
        self.n_shards = n_shards
        # テキスト n_shardsテキスト テキスト (テキスト "GPU"テキスト テキスト テキスト)
        params = list(model.parameters())
        self.shards = [params[i::n_shards] for i in range(n_shards)]

    def step(self, x, y, optimizer):
        """テキスト Step: Forward Pass → Backward Pass → テキスト shardテキスト Update."""
        optimizer.zero_grad()
        out = self.model(x)
        loss = F.mse_loss(out, y)
        loss.backward()

        # テキスト shardテキスト Gradientテキスト テキスト (テキスト shardテキスト 0テキスト)
        # テキスト FSDPテキスト テキスト テキスト, テキスト テキスト Gradient テキスト
        optimizer.step()
        return loss.item()

# テキスト
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 128)
)
fsdp = FSDPSimulator(model, n_shards=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(8, 128)
y = torch.randn(8, 128)
loss = fsdp.step(x, y, opt)
print(f"FSDP テキスト loss: {loss:.4f}")
print(f"テキスト テキスト: {sum(p.numel() for p in model.parameters()):,}")
print(f"Shard テキスト: {fsdp.n_shards} (テキスト shardテキスト ~{sum(p.numel() for p in model.parameters()) // 4:,} テキスト)")


## 30.8 要点

| Stage | テキスト テキスト | テキスト | テキスト |
|---|---|---|---|
| DDP | テキスト | $16P$ | All-Reduce |
| ZeRO-1 | テキスト | $16P/N + 4P$ | All-Reduce |
| ZeRO-2 | + テキスト | $16P/N + 2P$ | Reduce-Scatter |
| ZeRO-3 | + テキスト | $16P/N$ | All-Gather + Reduce-Scatter |
| FSDP | ZeRO-3 (PyTorch) | $16P/N$ | テキスト |
| ZeRO-Infinity | + CPU/NVMe | テキスト テキスト | テキスト |

## 演習問題

1. ZeRO Stage 0/1/2/3テキスト テキスト 70B モデル, N=16テキスト テキスト 計算テキスト.
2. FSDPテキスト 順伝播 テキスト All-Gatherテキスト テキスト テキスト テキスト.
3. ZeRO-3テキスト テキスト テキスト DDPテキスト 比較テキスト.
4. CPU Offloadingテキスト 速度テキスト テキスト テキスト テキスト テキスト.
5. 70B モデルテキスト 学習テキスト ZeRO-3 + テキスト テキスト A100 80GBテキスト テキスト 計算テキスト.

> 解答: `solutions/ch30_solutions.ipynb`
